## 1. Naives bayes classifier

In [9]:
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))

In [14]:
# --- BLOC 1 : charger le vrai dataset ---
from classical_ml.data_loader import load_movie_reviews
from classical_ml.naive_bayes_model import evaluate_model, train_naive_bayes

X_train, X_test, y_train, y_test = load_movie_reviews()
print("Taille train :", len(X_train), " | Taille test :", len(X_test))
print("Exemple d'avis (100 premiers caracteres) :", X_train[0][:200])

Taille train : 1600  | Taille test : 400
Exemple d'avis (100 premiers caracteres) : saving private ryan ( dreamworks ) running time : 2 hours 48 minutes . 
starring tom hanks , edward burns , tom sizemore and matt damon directed by steven spielberg already being hailed as the 'greate


In [ ]:
# --- BLOC 2 : entrainement et evaluation ---
model, vectorizer = train_naive_bayes(X_train, y_train)
resultats = evaluate_model(model, vectorizer, X_test, y_test)


print("\nAccuracy :", resultats["accuracy"])
print("\nRapport detaille :")
for classe, valeurs in resultats["report"].items():
    if isinstance(valeurs, dict):
        print(
            f"  {classe:12} precision={valeurs['precision']:.3f} j\
                recall={valeurs['recall']:.3f} f1={valeurs['f1-score']:.3f}"
        )


Accuracy : 0.8075

Rapport detaille :
  neg          precision=0.794 recall=0.830 f1=0.812
  pos          precision=0.822 recall=0.785 f1=0.803
  macro avg    precision=0.808 recall=0.807 f1=0.807
  weighted avg precision=0.808 recall=0.807 f1=0.807


In [21]:
# --- BLOC 3 : effet de max_features sur la performance ---
for taille in [500, 2000, 5000, 10000]:
    m, v = train_naive_bayes(X_train, y_train, max_features=taille)
    r = evaluate_model(m, v, X_test, y_test)
    print(f"max_features={taille:6} -> accuracy={r['accuracy']:.3f}")

max_features=   500 -> accuracy=0.757
max_features=  2000 -> accuracy=0.812
max_features=  5000 -> accuracy=0.807
max_features= 10000 -> accuracy=0.795


In [22]:
# --- BLOC 4 : inspecter les mots les plus "positifs" et "negatifs" selon le modele ---
import numpy as np

feature_names = np.array(vectorizer.get_feature_names_out())
log_probs = model.feature_log_prob_  # (nb_classes, nb_features)
classes = model.classes_
print("\nClasses dans l'ordre :", classes)

# difference de log-probabilite entre les deux classes, par mot
diff = log_probs[1] - log_probs[0]  # suppose classes[1] = 'pos', classes[0] = 'neg'
top_positifs = feature_names[np.argsort(diff)[-10:]]
top_negatifs = feature_names[np.argsort(diff)[:10]]

print("Mots les plus associes a 'pos' :", list(top_positifs))
print("Mots les plus associes a 'neg' :", list(top_negatifs))


Classes dans l'ordre : ['neg' 'pos']
Mots les plus associes a 'pos' : ['argento', 'hanks', 'lebowski', 'crowe', 'outstanding', 'damon', 'shrek', 'truman', 'flynt', 'mulan']
Mots les plus associes a 'neg' : ['seagal', 'worst', 'jawbreaker', 'waste', 'boring', 'ridiculous', 'schumacher', 'martha', 'stupid', 'lame']


In [25]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB

X_train = [
    "amazing great delivery",
    "amazing great delivery",
    "terrible worst delivery",
    "terrible worst delivery",
]
y_train = ["pos", "pos", "neg", "neg"]

vectorizer = TfidfVectorizer()
X_vec = vectorizer.fit_transform(X_train)
model = MultinomialNB()
model.fit(X_vec, y_train)

print("Vocabulaire (ordre des colonnes) :", list(vectorizer.get_feature_names_out()))
print("Classes apprises (ordre) :", model.classes_)
print()
print("feature_log_prob_ (shape) :", model.feature_log_prob_.shape)
print(model.feature_log_prob_)

Vocabulaire (ordre des colonnes) : ['amazing', 'delivery', 'great', 'terrible', 'worst']
Classes apprises (ordre) : ['neg' 'pos']

feature_log_prob_ (shape) : (2, 5)
[[-2.12936555 -1.51537334 -2.12936555 -1.30480943 -1.30480943]
 [-1.30480943 -1.51537334 -1.30480943 -2.12936555 -2.12936555]]


## 2. Logistic regression

In [3]:
# --- BLOC 1 : entrainement et comparaison directe avec Naive Bayes ---
import os
import sys

sys.path.insert(0, os.path.abspath(os.path.join("..", "src")))
from classical_ml.data_loader import load_movie_reviews
from classical_ml.logistic_regression import evaluate_model as eval_lr
from classical_ml.logistic_regression import train_logistic_regression
from classical_ml.naive_bayes_model import evaluate_model as eval_nb
from classical_ml.naive_bayes_model import train_naive_bayes

X_train, X_test, y_train, y_test = load_movie_reviews()

model_nb, vec_nb = train_naive_bayes(X_train, y_train)
resultats_nb = eval_nb(model_nb, vec_nb, X_test, y_test)

model_lr, vec_lr = train_logistic_regression(X_train, y_train)
resultats_lr = eval_lr(model_lr, vec_lr, X_test, y_test)

print(f"Naive Bayes         : {resultats_nb['accuracy']:.3f}")
print(f"Logistic Regression : {resultats_lr['accuracy']:.3f}")

Naive Bayes         : 0.807
Logistic Regression : 0.828


In [4]:
# --- BLOC 2 : effet du parametre C (regularisation) ---
from sklearn.metrics import accuracy_score

print("\n{:>8} {:>16} {:>16}".format("C", "accuracy TRAIN", "accuracy TEST"))
for C in [0.001, 0.01, 0.1, 1.0, 10, 100]:
    model, vectorizer = train_logistic_regression(X_train, y_train, C=C)
    X_train_vec = vectorizer.transform(X_train)
    acc_train = accuracy_score(y_train, model.predict(X_train_vec))
    resultats = eval_lr(model, vectorizer, X_test, y_test)
    print(f"{C:>8} {acc_train:>16.3f} {resultats['accuracy']:>16.3f}")


       C   accuracy TRAIN    accuracy TEST
   0.001            0.891            0.787
    0.01            0.887            0.792
     0.1            0.901            0.805
     1.0            0.962            0.828
      10            0.999            0.835
     100            1.000            0.830


In [ ]:
# --- BLOC 3 : inspecter les poids appris \
# (equivalent du feature_log_prob_ de Naive Bayes) ---
import numpy as np

model, vectorizer = train_logistic_regression(X_train, y_train, C=1.0)
feature_names = np.array(vectorizer.get_feature_names_out())
coefficients = model.coef_[0]  # un seul jeu de poids (classification binaire)

top_positifs = feature_names[np.argsort(coefficients)[-10:]]
top_negatifs = feature_names[np.argsort(coefficients)[:10]]

print(
    "\nMots avec le coefficient le plus POSITIF (poussent vers 'pos') :",
    list(top_positifs),
)
print(
    "Mots avec le coefficient le plus NEGATIF (poussent vers 'neg') :",
    list(top_negatifs),
)


Mots avec le coefficient le plus POSITIF (poussent vers 'pos') : ['mulan', 'overall', 'truman', 'perfect', 'best', 'family', 'excellent', 'war', 'life', 'great']
Mots avec le coefficient le plus NEGATIF (poussent vers 'neg') : ['bad', 'worst', 'plot', 'boring', 'movie', 'supposed', 'script', 'reason', 'waste', 'stupid']
